Transformer解析  
[论文](https://arxiv.org/abs/1706.03762)  
[参考](https://zhuanlan.zhihu.com/p/338817680)  
![总体架构](./imgs/transformer_all_arch.png)

# 1、输入处理
## 1.1 词嵌入
> 将输入prompt向量化，$d_model$表示每个词向量的维度
## 1.2 位置嵌入
> 由于进行attention score计算时，每个q都可以与所有K进行交互，没有顺序概念，需要添加时间信息

### 1.2.1 绝对位置编码
> transformer使用正余弦位置编码获取位置信息，然后通过向量相加的方式将位置信息与词信息结合。这里是在**词嵌入向量中加入位置信息**
$$PE_{i, 2t} = \sin (k/10000^{2t/d})$$
$$PE_{i, 2t+1} = \cos (k/10000^{2t/d})$$
其中$PE_{i, 2t}$ 表示d维度向量 $PE_i$中第2t位置分量

In [1]:
import torch

def sinusodial_position_encoding(seq_len, d_model) -> torch.Tensor:
    encoding = torch.zeros(seq_len, d_model)
    pos = torch.arange(0, seq_len)
    pos = pos.float().unsqueeze(dim=1) # 第dim维添加一个维度
    print(f"pos={pos}\n")

    _2i = torch.arange(0, d_model, step=2).float()
    print(f"_2i={_2i}\n")

    encoding[:, 0::2] = torch.sin(pos / (10000 ** (_2i / d_model)))
    encoding[:, 1::2] = torch.cos(pos / (10000 ** (_2i / d_model)))

    return encoding

seq_len = 5
d_model = 4
pe = sinusodial_position_encoding(seq_len, d_model)
print(pe)


pos=tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.]])

_2i=tensor([0., 2.])

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [-0.7568, -0.6536,  0.0400,  0.9992]])


> 存在可训练的绝对位置BERT，暂时不表

### 1.2.2 相对位置编码
理论知识参考[RoPE](https://zhuanlan.zhihu.com/p/642884818)

In [6]:
#！基于theta构建位置复数
import torch

def rope_params(max_seq_len, dim, theta=10000) :
    # max_seq_len=2, dim=4
    assert dim % 2 == 0
    # positions:tensor([0, 1])
    positions = torch.arange(max_seq_len)
    # thetas: tensor([1.0000, 0.0100], dtype=torch.float64)
    thetas = 1.0 / torch.pow(theta, torch.arange(0, dim, 2).to(torch.float64).div(dim))
    # freqs:tensor([[0.0000, 0.0000], [1.0000, 0.0100]], dtype=torch.float64)
    # 计算外积，结果为矩阵
    freqs = torch.outer(positions, thetas)
    
    # tensor([[1.0000+0.0000j, 1.0000+0.0000j], [0.5403+0.8415j, 1.0000+0.0100j]], dtype=torch.complex128)
    # real = abs * cos(theta) unreal = abs * sin(theta)
    freqs = torch.polar(torch.ones_like(freqs), freqs)

    return freqs

# d = dim // num_heads
d = 256
# 视频position分配
ffreqs = rope_params(1024, d - 4 * (d // 6))
hfreqs = rope_params(1024, 2 * (d // 6))
wfreqs = rope_params(1024, 2 * (d // 6))
# (1024, d)
freqs = torch.cat([ffreqs, hfreqs, wfreqs], dim=1) 


In [ ]:
import torch

def rope_apply(x, grid_sizes, freqs):
    # x.shape = B, L, num_heads, d_model / num_heads
    n, half_d = x.size(2), x.size(3) // 2

    # dim->f,h,w
    freqs = freqs.split([half_d - 2 * (half_d // 3), half_d // 3, half_d // 3], dim=1)

    output = []
    # batch loop
    for i, (f, h, w) in enumerate(grid_sizes.tolist()):
        seq_len = f * h * w
        # [seq_len, n, half_d]->[seq_len, n, half_d / 2, 2]
        x_i = x[i, :seq_len].to(torch.float64).reshape(seq_len, n, -1, 2)
        x_i_complex = torch.view_as_complex(x_i)
        # [seq_len, f] -> [f, 1, 1, seq_len] -> [f, h, w, seq_len]
        ffreqs = freqs[0][:f].view(f, 1, 1, -1).expand(f, h, w, -1)
        # [seq_len, h] -> [1, h, 1, seq_len] -> [f, h, w, seq_len]
        hfreqs = freqs[1][:h].view(1, h, 1, -1).expand(f, h, w, -1)
        # [seq_len, w] -> [1, 1, w, seq_len] -> [f, h, w, seq_len]
        wfreqs = freqs[2][:w].view(1, 1, w, -1).expand(f, h, w, -1)

        # [f, h, w, seq_len] -> [seq_len, 1, f * h * w]
        freqs_i = torch.cat(
            [ffreqs, hfreqs, wfreqs], dim=-1
        ).reshape(seq_len, 1, -1)

        x_i = torch.view_as_real(x_i_complex * freqs_i).flatten(2, -1)
        # 填充
        x_i = torch.cat([x_i, x[i, seq_len:]])

        output.append(x_i)

    return torch.stack(output).float()





# 2、注意力机制
![self-attention](./imgs/scaled_dot-product_attention.png)

> <mark>Q/K/V为什么要经过Linear层后才能进行计算?</mark>

得到Q/K/V后基于以下公式计算出self-attention score
$$ Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}}) V $$

其中$d_k$为词向量维度。

> <mark>为什么要除以d_k的开方？</mark>

In [7]:
import math
import torch.nn as nn

class SacleDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaleDotProductAttention, self).__init__()
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, q, k, v, mask=None, e=1e-12):
        batch_size, head, seq_len, d_tensor = k.size()

        # 1. dot product Query with Key^T to compute similarity
        k_t = k.transpose(2, 3)
        score = (q @ k_t) / math.sqrt(d_tensor)

        # 2. apply masking (optional)
        if mask is not None:
            score = score.masked_fill(mask == 0, -10000)
        
        # 3. pass them softmax to make [0, 1] range
        score = self.softmax(score)

        # 4. multiply with Value
        # 按照矩阵的方式相乘即torch.matmul(score, v)
        v = score @ v

        return v, score

## 2.1 多头注意力（Multi-head Attention）
> 将d_k维度分为(head_num, d_k / head_num)。**多头注意力的核心价值**：通过多个并行的注意力机制，让模型能够同时关注不同位置、不同维度的特征，捕捉更丰富的语言现象，从而提高表示能力。这就像用一个专家团队代替一个全能专家，每个专家专注于自己擅长的领域，最终形成更全面的理解。


In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super(MultiHeadAttention, self).__init__()
        self.n_head = n_head
        self.d_model = d_model
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.attention = SacleDotProductAttention()

        self.w_concat = nn.Linear(d_model, d_model)

    def forward(self, q, k , v, mask=None):
        # 1. 点乘权重矩阵
        q, k, v = self.w_q(q), self.w_k(k), self.w_v(v)

        # 2. 获取多头
        q, k, v = self._split(q), self._split(k), self._split(v)

        # 3. 计算相似度
        out, attention = self.attention(q, k, v, mask=mask)

        # 4. concat & pass to linear
        out = self._concat(out)
        out = self.w_concat(out)

        return out

    def _split(self, tensor):
        # [batch_size, seq_len, d_model] -> [batch_size, n_head, seq_len, d_k]
        b, s, d = tensor.size()
        d_tensor = d // self.n_head
        tensor = tensor.view(b, s, self.n_head, d_tensor).transpose(1, 2)
        return tensor

    def _concat(self, tensor):
        batch_size, head_num, seq_len, d_k = tensor.size()
        d_model = head_num * d_k

        tensor = tensor.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        return tensor